# Healthcare Lab 1: Clinical NLP Agent -- Extracting Structure from Clinical Notes

**Series**: Agentic AI Science Playbook -- Healthcare Domain
**Goal**: Build an agent that extracts structured medical information (medications, diagnoses, vitals) from unstructured clinical notes.
**Prerequisites**: Foundation Labs 0-2.
**Time**: ~60 min.

---

## Background: Clinical NLP

Electronic Health Records (EHRs) contain a wealth of information in free-text clinical notes -- physician summaries, discharge summaries, nursing notes. Extracting structured information from this text is one of the most impactful applications of NLP in healthcare.

### Key Extraction Tasks

| Task | Example |
|------|---------|
| **Medications** | Drug name, dose, frequency, route, status (active/discontinued) |
| **Diagnoses** | ICD-10 codes or free-text conditions with status (present/historical/ruled-out) |
| **Vitals** | Blood pressure, heart rate, temperature, SpO2, weight |
| **Procedures** | Surgeries, imaging studies, lab tests ordered |
| **Allergies** | Drug/substance with reaction type |

### Important disclaimer
This lab uses synthetic/de-identified clinical notes for educational purposes. Never use real patient data without appropriate IRB approval and data use agreements.

---

## What You Will Build

A clinical NLP agent with three tools:
- `extract_medications` -- extract drug names, doses, routes, and status
- `extract_diagnoses` -- extract conditions with ICD hints and clinical status
- `extract_vitals` -- extract numerical measurements with units

## Learning Objectives

By the end of this lab, you will be able to:
- Build agents that extract structured data from unstructured clinical text
- Implement medication, diagnosis, and vital sign extraction tools
- Design safety-critical features like allergy conflict detection
- Understand the unique challenges of clinical NLP

## Why This Matters

Healthcare generates ~30% of the world's data, mostly as unstructured clinical notes. Manually reviewing these notes is the #1 time sink for healthcare professionals.

Clinical NLP agents can:
- **Reduce documentation burden** by auto-extracting structured data
- **Improve patient safety** by detecting drug interactions automatically
- **Accelerate research** by making clinical data searchable and analyzable

> **Safety note**: Clinical NLP agents are assistive tools — they augment clinicians, never replace clinical judgment. This lab teaches the pattern of **conservative extraction** with built-in safety checks.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | Foundation Labs 0-2 |
| Domain knowledge | None required — clinical concepts explained inline |

**NVIDIA Connection**: NVIDIA's healthcare AI ecosystem includes **Clara** for medical imaging and **BioNeMo** for molecular biology. The clinical NLP patterns you learn here integrate with NVIDIA's enterprise healthcare platform, where **NIM** endpoints serve medical-grade language models with HIPAA-compliant deployment options.

### Install Dependencies
Install the required Python packages for this lab. We need `openai` for LLM access and `pydantic` for data validation of our tool schemas.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Set up your OpenAI or NVIDIA NIM connection. This cell detects which API you have configured and creates the client. It also imports the core libraries we will use throughout the lab.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")
import json
from pydantic import BaseModel, Field
from typing import Literal

## Synthetic Clinical Notes (De-identified)

## Concept: Structured Extraction from Clinical Text

Clinical NLP extraction follows a consistent pattern:

```
Unstructured text → LLM extraction → Structured JSON → Pydantic validation → Application
```

The key design decisions:
- **Use `response_format={"type": "json_object"}`** to ensure parseable output
- **Define clear extraction schemas** for each entity type (medications, diagnoses, vitals)
- **Include status fields** (active/discontinued/held) because clinical context matters
- **Conservative extraction**: only extract what is clearly stated (never infer)

### Load Synthetic Clinical Notes
Two de-identified clinical notes: a discharge summary and a progress note. These are realistic but entirely synthetic -- no real patient data. The discharge summary includes medications, diagnoses, vitals, and allergies. The progress note covers a post-operative recovery.

In [ ]:
SAMPLE_NOTES = {
    "discharge_summary": """
DISCHARGE SUMMARY
Patient: [REDACTED], DOB: [REDACTED]
Admission: [REDACTED] | Discharge: [REDACTED]

CHIEF COMPLAINT: Chest pain and shortness of breath

DIAGNOSES:
1. Acute myocardial infarction (NSTEMI), confirmed by troponin elevation
2. Type 2 diabetes mellitus, poorly controlled (HbA1c 9.2%)
3. Hypertension, essential
4. Hyperlipidemia

MEDICATIONS ON DISCHARGE:
- Aspirin 81mg PO daily
- Metoprolol succinate 50mg PO daily
- Lisinopril 10mg PO daily
- Atorvastatin 40mg PO nightly
- Metformin 1000mg PO twice daily (held during admission, resumed at discharge)
- Insulin glargine 20 units SC at bedtime (new)

VITALS AT DISCHARGE:
BP: 128/78 mmHg | HR: 68 bpm | RR: 16 breaths/min | Temp: 36.8°C | SpO2: 98% on room air
Weight: 87 kg

ALLERGIES: Penicillin (anaphylaxis), Sulfa (rash)
""",

    "progress_note": """
PROGRESS NOTE - Day 3 Post-op
Patient s/p laparoscopic appendectomy

Subjective: Patient reports pain 3/10, tolerating clear liquids. No nausea/vomiting.
No fever reported.

Objective:
Vitals: T 37.1°C, HR 82 bpm, BP 118/72 mmHg, RR 14, SpO2 99% RA
Labs: WBC 11.2 K/uL (down from 18.5), CRP 45 mg/L (down from 120)

Assessment:
1. Status post laparoscopic appendectomy, recovering well
2. Post-operative infection risk decreasing, WBC trending down
3. Pain adequately controlled

Plan:
- Continue ketorolac 15mg IV q6h x 1 more day, then transition to ibuprofen 400mg PO q8h PRN
- Advance diet to regular as tolerated
- Discontinue IV access if no fever tonight
- Discharge planning: follow-up in 2 weeks
""",
}
print("Sample notes loaded:", list(SAMPLE_NOTES.keys()))

## Tool Schemas

### Define Clinical NLP Tool Schemas
Three extraction tools, each with specific parameters. Note `include_discontinued` and `normalize_units` -- these control extraction behavior for clinical accuracy. The schemas tell the LLM exactly what arguments each tool accepts.

In [ ]:
class ExtractMedicationsArgs(BaseModel):
    note_text: str = Field(..., description="Clinical note text to extract medications from.")
    include_discontinued: bool = Field(True, description="Include discontinued or held medications.")

class ExtractDiagnosesArgs(BaseModel):
    note_text: str = Field(..., description="Clinical note text to extract diagnoses from.")
    include_historical: bool = Field(True, description="Include historical/past diagnoses.")

class ExtractVitalsArgs(BaseModel):
    note_text: str = Field(..., description="Clinical note text to extract vital signs from.")
    normalize_units: bool = Field(True, description="Normalize all values to standard SI units.")

OPENAI_TOOLS = [
    {"type": "function", "function": {
        "name": "extract_medications",
        "description": "Extract medication information from a clinical note. Returns drug names, doses, routes, frequencies, and status.",
        "parameters": ExtractMedicationsArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "extract_diagnoses",
        "description": "Extract diagnoses and conditions from a clinical note with clinical status.",
        "parameters": ExtractDiagnosesArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "extract_vitals",
        "description": "Extract vital signs and measurements from a clinical note.",
        "parameters": ExtractVitalsArgs.model_json_schema()}},
]
SCHEMA_MAP = {
    "extract_medications": ExtractMedicationsArgs,
    "extract_diagnoses": ExtractDiagnosesArgs,
    "extract_vitals": ExtractVitalsArgs,
}
print("Clinical NLP tools:", [t["function"]["name"] for t in OPENAI_TOOLS])

## Tool Implementations (LLM-powered)

### Implement LLM-Powered Extraction
Each tool sends the clinical note to the LLM with a structured extraction prompt. The key: `response_format={'type': 'json_object'}` ensures we get parseable structured data back. We also define the agent function and tool dispatcher here.

In [ ]:
def execute_extract_medications(args: ExtractMedicationsArgs) -> str:
    prompt = (
        "Extract all medications from this clinical note. "
        "For each medication return: name, dose, route, frequency, status (active/held/discontinued/new).\n"
        "Return as JSON: {\"medications\": [{\"name\": ..., \"dose\": ..., \"route\": ..., \"frequency\": ..., \"status\": ...}]}\n"
        f"Include discontinued: {args.include_discontinued}\n"
        f"Note:\n{args.note_text[:2000]}"
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=800,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content or "{}"

def execute_extract_diagnoses(args: ExtractDiagnosesArgs) -> str:
    prompt = (
        "Extract all diagnoses/conditions from this clinical note. "
        "For each: condition name, status (active/resolved/historical/ruled_out), and confidence (high/moderate/low).\n"
        "Return as JSON: {\"diagnoses\": [{\"condition\": ..., \"status\": ..., \"confidence\": ...}]}\n"
        f"Note:\n{args.note_text[:2000]}"
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=600,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content or "{}"

def execute_extract_vitals(args: ExtractVitalsArgs) -> str:
    prompt = (
        "Extract all vital signs and measurements from this clinical note. "
        "For each: measurement name, value, unit, and time point if specified.\n"
        "Return as JSON: {\"vitals\": [{\"name\": ..., \"value\": ..., \"unit\": ..., \"time_point\": ...}]}\n"
        f"Note:\n{args.note_text[:2000]}"
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=400,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content or "{}"

def run_tool(name, args):
    if name == "extract_medications": return execute_extract_medications(args)
    if name == "extract_diagnoses": return execute_extract_diagnoses(args)
    if name == "extract_vitals": return execute_extract_vitals(args)
    return f"[error] Unknown: {name}"

CLINICAL_SYSTEM = (
    "You are a clinical NLP assistant. Extract structured medical information from clinical notes "
    "using the provided tools. Be precise and conservative -- only extract what is clearly stated."
)

def clinical_agent(user_message):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[{"role": "system", "content": CLINICAL_SYSTEM},
                   {"role": "user", "content": user_message}],
        tools=OPENAI_TOOLS, tool_choice="auto")
    msg = r.choices[0].message
    if not msg.tool_calls:
        return {"tool": None, "content": msg.content}
    call = msg.tool_calls[0]
    validated = SCHEMA_MAP[call.function.name](**json.loads(call.function.arguments))
    return {"tool": call.function.name, "args": validated}

## Experiment 1: Extract from Discharge Summary

### Experiment 1a: Extract Medications
Extract all medications from the discharge summary. The agent should find 6 medications with dose, route, frequency, and status. This tests whether the LLM can correctly identify active, held, and newly-started medications.

In [ ]:
note = SAMPLE_NOTES["discharge_summary"]

# Extract medications
r = clinical_agent(f"Extract all medications from this discharge summary: {note}")
if r["tool"]:
    result = json.loads(run_tool(r["tool"], r["args"]))
    print("MEDICATIONS:")
    for med in result.get("medications", []):
        print(f"  {med.get('name','?')}: {med.get('dose','?')} {med.get('route','?')} {med.get('frequency','?')} [{med.get('status','?')}]")

<details>
<summary>Expected output (click to expand)</summary>

```
MEDICATIONS:
  Aspirin: 81mg PO daily [active]
  Metoprolol succinate: 50mg PO daily [active]
  Lisinopril: 10mg PO daily [active]
  Atorvastatin: 40mg PO nightly [active]
  Metformin: 1000mg PO twice daily [resumed]
  Insulin glargine: 20 units SC at bedtime [new]
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Experiment 1b: Extract Diagnoses
Extract conditions with clinical status (active/resolved/historical). The discharge summary lists 4 diagnoses including an acute MI and poorly controlled diabetes. The agent should capture each with its clinical context and confidence level.

In [ ]:
# Extract diagnoses
r = clinical_agent(f"Extract all diagnoses from this discharge summary: {note}")
if r["tool"]:
    result = json.loads(run_tool(r["tool"], r["args"]))
    print("DIAGNOSES:")
    for dx in result.get("diagnoses", []):
        print(f"  {dx.get('condition','?')} -- status: {dx.get('status','?')} (confidence: {dx.get('confidence','?')})")

<details>
<summary>Expected output (click to expand)</summary>

```
DIAGNOSES:
  Acute myocardial infarction (NSTEMI) -- status: active (confidence: high)
  Type 2 diabetes mellitus -- status: active (confidence: high)
  Hypertension, essential -- status: active (confidence: high)
  Hyperlipidemia -- status: active (confidence: high)
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Experiment 1c: Extract Vital Signs
Extract numerical measurements with units. The discharge summary contains blood pressure, heart rate, respiratory rate, temperature, SpO2, and weight. The agent should parse each value with its correct unit.

In [ ]:
# Extract vitals
r = clinical_agent(f"Extract vital signs from this discharge summary: {note}")
if r["tool"]:
    result = json.loads(run_tool(r["tool"], r["args"]))
    print("VITALS:")
    for v in result.get("vitals", []):
        print(f"  {v.get('name','?')}: {v.get('value','?')} {v.get('unit','?')}")

<details>
<summary>Expected output (click to expand)</summary>

```
VITALS:
  Blood pressure: 128/78 mmHg
  Heart rate: 68 bpm
  Respiratory rate: 16 breaths/min
  Temperature: 36.8 °C
  SpO2: 98 %
  Weight: 87 kg
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Experiment 2: Extract from Progress Note

### Experiment 2: Progress Note Extraction
Test all three tools on a different note type -- a post-operative progress note. This note has different structure and vocabulary compared to the discharge summary. We run medication, diagnosis, and vital/lab extraction in a single loop to see how the tools handle a surgical context.

In [ ]:
note2 = SAMPLE_NOTES["progress_note"]

for request, label in [
    (f"Extract all medications from this progress note: {note2}", "MEDICATIONS"),
    (f"Extract diagnoses and problems from this progress note: {note2}", "DIAGNOSES"),
    (f"Extract vital signs and lab values from this progress note: {note2}", "VITALS/LABS"),
]:
    r = clinical_agent(request)
    if r["tool"]:
        result = json.loads(run_tool(r["tool"], r["args"]))
        print(f"\n{label}:")
        for key, items in result.items():
            for item in items:
                print(f"  {item}")

<details>
<summary>Expected output (click to expand)</summary>

```
MEDICATIONS:
  {'name': 'Ketorolac', 'dose': '15mg', 'route': 'IV', 'frequency': 'q6h', 'status': 'active'}
  {'name': 'Ibuprofen', 'dose': '400mg', 'route': 'PO', 'frequency': 'q8h PRN', 'status': 'planned'}

DIAGNOSES:
  {'condition': 'Status post laparoscopic appendectomy', 'status': 'active', 'confidence': 'high'}
  {'condition': 'Post-operative infection risk', 'status': 'active', 'confidence': 'moderate'}

VITALS/LABS:
  {'name': 'Temperature', 'value': 37.1, 'unit': '°C'}
  {'name': 'Heart rate', 'value': 82, 'unit': 'bpm'}
  {'name': 'Blood pressure', 'value': '118/72', 'unit': 'mmHg'}
  {'name': 'WBC', 'value': 11.2, 'unit': 'K/uL'}
  {'name': 'CRP', 'value': 45, 'unit': 'mg/L'}
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Concept: Safety Layers in Clinical Agents

Clinical agents require safety checks that go beyond standard agent patterns:

| Safety Layer | What It Catches | Implementation |
|-------------|----------------|----------------|
| **Allergy conflicts** | Drug prescribed to allergic patient | Cross-reference meds ↔ allergies |
| **Dose validation** | Dangerous dosages | Compare against known safe ranges |
| **Interaction checking** | Drug-drug interactions | Cross-reference medication pairs |
| **Confidence flagging** | Low-confidence extractions | Flag for human review |

> **Design principle**: Safety checks should be **separate tools**, not embedded in extraction. This lets you update safety rules without changing the extraction logic.

### Experiment 3: Allergy Conflict Detection
A safety check: cross-reference extracted medications against the patient's documented allergies to flag potential conflicts. This is a separate post-processing step that demonstrates the safety layer pattern -- extraction and safety checking are intentionally kept as independent operations.

## Experiment 3: Safety Check -- Allergy Conflict Detection

In [ ]:
def check_allergy_conflicts(medications_json: str, note_text: str) -> str:
    prompt = (
        f"Given these extracted medications: {medications_json}\n"
        f"And this clinical note (which may contain allergy information): {note_text[:1000]}\n"
        "Identify any potential allergy conflicts or drug interactions. "
        "Return JSON: {\"conflicts\": [{\"drug\": ..., \"allergen\": ..., \"severity\": ..., \"recommendation\": ...}]}"
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=400,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content or "{}"

note = SAMPLE_NOTES["discharge_summary"]
r = clinical_agent(f"Extract all medications from this note: {note}")
meds_json = run_tool(r["tool"], r["args"]) if r["tool"] else "{}"
conflicts = json.loads(check_allergy_conflicts(meds_json, note))
print("ALLERGY CONFLICT CHECK:")
for conflict in conflicts.get("conflicts", []):
    print(f"  ALERT: {conflict}")
if not conflicts.get("conflicts"):
    print("  No conflicts detected.")

<details>
<summary>Expected output (click to expand)</summary>

```
ALLERGY CONFLICT CHECK:
  No conflicts detected.
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The agent checks for penicillin and sulfa allergy conflicts. The discharge medications (aspirin, metoprolol, lisinopril, atorvastatin, metformin, insulin glargine) should not trigger these allergies.
</details>

## Reflection Questions

1. **The agent extracted medications from synthetic notes.** What additional challenges would arise with real clinical notes (abbreviations, typos, negation)?
2. **Allergy conflict detection is a post-processing step.** Why is it important to separate extraction from safety checking?
3. **How would you evaluate** the accuracy of clinical NLP extraction? What metrics (precision, recall, F1) matter most for patient safety?

## Summary

| Tool | Capability |
|------|-----------|
| `extract_medications` | Drug name, dose, route, frequency, status |
| `extract_diagnoses` | Conditions with status and confidence |
| `extract_vitals` | Measurements with units and time points |

**Key lessons**:
- LLM-powered extraction is flexible but requires careful prompting and validation
- `response_format={"type": "json_object"}` gives structured, parseable output
- Always implement safety checks (allergy conflicts, dose ranges) as a separate verification step

**Next**: HC Lab 2 -- Medical Literature Agent for searching and summarizing scientific evidence.

---
*Agentic AI Science Playbook -- Healthcare Domain, Lab HC1.*

**Disclaimer**: This lab uses synthetic de-identified notes for educational purposes only.